# MV CAF Backbone Test v1.1

`baseline v2.1`을 바탕으로 CNN, ConvNeXt, Swin, ViT 계열 backbone을 비교하는 multi-view cross-attention 실험 notebook입니다. dev 기준 early stopping, TTA, test inference, weighted ensemble까지 수행합니다.

In [1]:
from __future__ import annotations

import json
import os
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from IPython.display import display

ROOT = Path.cwd().resolve().parent
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from augmentations import build_default_transforms
from models import EMAConfig, ModelEMA
from reproducibility import make_generator, seed_everything, seed_worker

DATA_DIR = ROOT / "data"
EXPERIMENT_NAME = "mv_caf_backbone_test"
EXPERIMENT_VERSION = "v1.2"
WEIGHT_DIR = ROOT / "outputs" / "weights" / EXPERIMENT_NAME / EXPERIMENT_VERSION
SUBMISSION_DIR = ROOT / "outputs" / "submissions" / EXPERIMENT_NAME / EXPERIMENT_VERSION
EDA_DIR = ROOT / "outputs" / "eda_preprocessing" / EXPERIMENT_NAME / EXPERIMENT_VERSION
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
EDA_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    "IMG_SIZE": 224,
    "EPOCHS": 50,
    "EARLY_STOPPING_PATIENCE": 5,
    "LEARNING_RATE": 3e-4,
    "BATCH_SIZE": 16,
    "SEED": 42,
    "NUM_WORKERS": 8,
    "WEIGHT_DECAY": 1e-4,
    "MIXUP_ALPHA": 0.1,
    "MIXUP_PROB": 0.0,
    "MIN_LR": 1e-6,
    "EMA_DECAY": 0.999,
    "EMA_USE_FOR_EVAL": True,
    "TTA_CANDIDATES": [
        ["none"],
        ["none", "hflip"],
        ["none", "hflip", "crop95"],
    ],
    "VIDEO_AUG_ENABLE": True,
    "VIDEO_AUG_CACHE": True,
    "UNSTABLE_START_MIN_SEC": 0.5,
    "UNSTABLE_START_MAX_SEC": 1.0,
    "UNSTABLE_FRAMES_MIN": 2,
    "UNSTABLE_FRAMES_MAX": 3,
    "STABLE_END_MIN_SEC": 9.0,
    "STABLE_END_MAX_SEC": 10.0,
    "STABLE_FRAMES_PER_VIDEO": 2,
}

BACKBONE_CANDIDATES = [
    "efficientnetv2_rw_s",
    "convnext_tiny.fb_in22k_ft_in1k",
    "convnext_small.fb_in22k_ft_in1k",
    "swin_tiny_patch4_window7_224.ms_in22k_ft_in1k",
    "vit_small_patch16_224.augreg_in21k_ft_in1k",
    "vit_base_patch16_224.augreg_in21k_ft_in1k",
    "deit3_small_patch16_224.fb_in22k_ft_in1k",
    "vit_base_patch14_dinov2.lvd142m",
]

selected_backbones = []
for name in BACKBONE_CANDIDATES:
    try:
        timm.create_model(name, pretrained=False, num_classes=0, global_pool="")
        selected_backbones.append(name)
    except Exception as exc:
        print("skip backbone:", name, exc)

print("selected_backbones:", selected_backbones)
assert selected_backbones, "No candidate backbones are available in this timm installation."

seed_everything(CFG["SEED"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
            


selected_backbones: ['efficientnetv2_rw_s', 'convnext_tiny.fb_in22k_ft_in1k', 'convnext_small.fb_in22k_ft_in1k', 'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k', 'vit_small_patch16_224.augreg_in21k_ft_in1k', 'vit_base_patch16_224.augreg_in21k_ft_in1k', 'deit3_small_patch16_224.fb_in22k_ft_in1k', 'vit_base_patch14_dinov2.lvd142m']
device: cuda


In [2]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {"stable": 0, "unstable": 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = str(row["id"])
        base_dir = self.root_dir
        if ("sample_dir" in self.df.columns) and pd.notna(row.get("sample_dir", np.nan)):
            base_dir = str(row["sample_dir"])
        folder_path = os.path.join(base_dir, sample_id)

        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)

        if self.is_test:
            return views

        return views, self.label_map[row["label"]]


def _extract_frame_by_sec(cap, sec, fps, frame_count):
    frame_idx = int(max(0, min(frame_count - 1, round(sec * fps))))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _extract_last_frame(cap, frame_count):
    last_idx = max(0, frame_count - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, last_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _video_aug_cache_signature(cfg):
    keys = [
        "SEED",
        "UNSTABLE_START_MIN_SEC",
        "UNSTABLE_START_MAX_SEC",
        "UNSTABLE_FRAMES_MIN",
        "UNSTABLE_FRAMES_MAX",
        "STABLE_END_MIN_SEC",
        "STABLE_END_MAX_SEC",
        "STABLE_FRAMES_PER_VIDEO",
    ]
    return {k: cfg.get(k) for k in keys}


def build_video_augmented_df(train_df, data_dir, cfg):
    train_root = data_dir / "train"
    aug_root = data_dir / "train_video_aug"
    aug_root.mkdir(parents=True, exist_ok=True)

    cache_csv = aug_root / "aug_df.csv"
    cache_meta = aug_root / "cache_meta.json"
    cache_sig = _video_aug_cache_signature(cfg)

    if cfg.get("VIDEO_AUG_CACHE", True) and cache_csv.exists() and cache_meta.exists():
        try:
            meta = json.loads(cache_meta.read_text())
            if meta.get("signature") == cache_sig:
                cached_df = pd.read_csv(cache_csv)
                if not cached_df.empty:
                    sample_ids = cached_df["id"].astype(str).tolist()
                    missing_ids = [
                        sample_id
                        for sample_id in sample_ids
                        if not (aug_root / sample_id / "front.png").exists()
                        or not (aug_root / sample_id / "top.png").exists()
                    ]
                    if not missing_ids:
                        cached_df["sample_dir"] = str(aug_root)
                        return cached_df
                    print(
                        "video aug cache invalid: missing rendered files for",
                        len(missing_ids),
                        "samples. rebuilding cache.",
                    )
        except Exception as exc:
            print("video aug cache read failed:", exc)

    for p in aug_root.glob("AUGV_*"):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)

    rng = np.random.default_rng(cfg["SEED"])
    stable_rows = []
    unstable_rows = []
    saved_idx = 0

    def save_aug(img, label):
        nonlocal saved_idx
        aug_id = f"AUGV_{saved_idx:07d}"
        out_dir = aug_root / aug_id
        out_dir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(img).save(out_dir / "front.png")
        Image.fromarray(img).save(out_dir / "top.png")
        row = {"id": aug_id, "label": label, "sample_dir": str(aug_root)}
        saved_idx += 1
        return row

    for sample_id in tqdm(train_df["id"].tolist(), desc="Video aug stable(last-frame)", dynamic_ncols=True, ascii=True):
        video_path = train_root / sample_id / "simulation.mp4"
        if not video_path.exists():
            continue
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            cap.release()
            continue
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if frame_count <= 0:
            cap.release()
            continue
        img = _extract_last_frame(cap, frame_count)
        cap.release()
        if img is not None:
            stable_rows.append(save_aug(img, "stable"))

    unstable_ids = train_df.loc[train_df["label"] == "unstable", "id"].tolist()
    for sample_id in tqdm(unstable_ids, desc="Video aug unstable(0.5~1.0s)", dynamic_ncols=True, ascii=True):
        video_path = train_root / sample_id / "simulation.mp4"
        if not video_path.exists():
            continue
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            cap.release()
            continue
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if fps is None or fps <= 0 or frame_count <= 1:
            cap.release()
            continue
        duration = frame_count / fps
        low = cfg["UNSTABLE_START_MIN_SEC"]
        high = min(cfg["UNSTABLE_START_MAX_SEC"], max(0.0, duration - 1.0 / fps))
        if high <= low:
            cap.release()
            continue
        n_unstable = int(rng.integers(cfg["UNSTABLE_FRAMES_MIN"], cfg["UNSTABLE_FRAMES_MAX"] + 1))
        unstable_secs = rng.uniform(low, high, size=n_unstable)
        for sec in unstable_secs:
            img = _extract_frame_by_sec(cap, float(sec), fps, frame_count)
            if img is not None:
                unstable_rows.append(save_aug(img, "unstable"))
        cap.release()

    stable_df = pd.DataFrame(stable_rows)
    unstable_df = pd.DataFrame(unstable_rows)
    if stable_df.empty or unstable_df.empty:
        return pd.DataFrame(columns=["id", "label", "sample_dir"])

    target_unstable = len(stable_df)
    if len(unstable_df) >= target_unstable:
        unstable_bal = unstable_df.sample(n=target_unstable, random_state=cfg["SEED"])
    else:
        unstable_bal = unstable_df.sample(n=target_unstable, replace=True, random_state=cfg["SEED"])

    aug_df = pd.concat([stable_df, unstable_bal], ignore_index=True)
    aug_df = aug_df.sample(frac=1.0, random_state=cfg["SEED"]).reset_index(drop=True)
    aug_df.to_csv(cache_csv, index=False)
    cache_meta.write_text(json.dumps({"signature": cache_sig}, ensure_ascii=False, indent=2))
    return aug_df
            


In [3]:
@dataclass(frozen=True)
class FlexibleCAFConfig:
    backbone_name: str = "efficientnetv2_rw_s"
    pretrained: bool = True
    attn_dim: int = 256
    num_heads: int = 8
    num_layers: int = 2
    dropout: float = 0.1
    classifier_hidden_dim: int = 512
    classifier_mid_dim: int = 128
    classifier_dropout: float = 0.3
    out_dim: int = 1


class CrossAttentionBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)

    def forward(self, q_tokens: torch.Tensor, kv_tokens: torch.Tensor) -> torch.Tensor:
        q = self.norm_q(q_tokens)
        kv = self.norm_kv(kv_tokens)
        attn_out, _ = self.attn(q, kv, kv, need_weights=False)
        return q_tokens + attn_out


class MultiViewFlexibleCAF(nn.Module):
    def __init__(self, config: FlexibleCAFConfig):
        super().__init__()
        self.config = config
        self.backbone = timm.create_model(
            config.backbone_name,
            pretrained=config.pretrained,
            num_classes=0,
            global_pool="",
        )
        self.feature_dim = getattr(self.backbone, "num_features")
        self.cnn_proj = nn.Conv2d(self.feature_dim, config.attn_dim, kernel_size=1, bias=False)
        self.token_proj = nn.Linear(self.feature_dim, config.attn_dim, bias=False)
        self.cross_12 = nn.ModuleList(
            [CrossAttentionBlock(config.attn_dim, config.num_heads, config.dropout) for _ in range(config.num_layers)]
        )
        self.cross_21 = nn.ModuleList(
            [CrossAttentionBlock(config.attn_dim, config.num_heads, config.dropout) for _ in range(config.num_layers)]
        )
        self.classifier = nn.Sequential(
            nn.Linear(config.attn_dim * 2, config.classifier_hidden_dim),
            nn.BatchNorm1d(config.classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(config.classifier_dropout),
            nn.Linear(config.classifier_hidden_dim, config.classifier_mid_dim),
            nn.ReLU(),
            nn.Linear(config.classifier_mid_dim, config.out_dim),
        )

    def _to_tokens(self, feat: torch.Tensor) -> torch.Tensor:
        if feat.ndim == 4:
            if feat.shape[1] == self.feature_dim:
                feat = self.cnn_proj(feat)
                return feat.flatten(2).transpose(1, 2)
            if feat.shape[-1] == self.feature_dim:
                feat = feat.permute(0, 3, 1, 2)
                feat = self.cnn_proj(feat)
                return feat.flatten(2).transpose(1, 2)
        if feat.ndim == 3:
            if feat.shape[-1] == self.feature_dim:
                return self.token_proj(feat)
            if feat.shape[1] == self.feature_dim:
                return self.token_proj(feat.transpose(1, 2))
        raise ValueError(f"Unsupported feature shape: {tuple(feat.shape)} for backbone={self.config.backbone_name}")

    def forward(self, views):
        f1 = self.backbone.forward_features(views[0])
        f2 = self.backbone.forward_features(views[1])
        t1 = self._to_tokens(f1)
        t2 = self._to_tokens(f2)

        for blk12, blk21 in zip(self.cross_12, self.cross_21):
            t1_prev, t2_prev = t1, t2
            t1 = blk12(t1_prev, t2_prev)
            t2 = blk21(t2_prev, t1_prev)

        feat = torch.cat([t1.mean(dim=1), t2.mean(dim=1)], dim=1)
        return self.classifier(feat)
            


In [4]:
train_transform, test_transform = build_default_transforms(CFG["IMG_SIZE"])

train_df = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")
val_df = pd.read_csv(DATA_DIR / "dev.csv", encoding="utf-8-sig")
test_df = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")

train_df_for_train = train_df.copy()
train_df_for_train["sample_dir"] = str(DATA_DIR / "train")

if CFG["VIDEO_AUG_ENABLE"]:
    aug_df = build_video_augmented_df(train_df, DATA_DIR, CFG)
    if not aug_df.empty:
        train_df_for_train = pd.concat([train_df_for_train, aug_df], ignore_index=True)

val_df_for_eval = val_df.copy()
val_df_for_eval["sample_dir"] = str(DATA_DIR / "dev")
test_df_for_infer = test_df.copy()
test_df_for_infer["sample_dir"] = str(DATA_DIR / "test")

train_dataset = MultiViewDataset(train_df_for_train, str(DATA_DIR / "train"), train_transform, is_test=False)
val_dataset = MultiViewDataset(val_df_for_eval, str(DATA_DIR / "dev"), test_transform, is_test=False)
test_dataset = MultiViewDataset(test_df_for_infer, str(DATA_DIR / "test"), test_transform, is_test=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["BATCH_SIZE"],
    shuffle=True,
    num_workers=CFG["NUM_WORKERS"],
    pin_memory=(device.type == "cuda"),
    worker_init_fn=seed_worker,
    generator=make_generator(CFG["SEED"]),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG["BATCH_SIZE"],
    shuffle=False,
    num_workers=CFG["NUM_WORKERS"],
    pin_memory=(device.type == "cuda"),
    worker_init_fn=seed_worker,
    generator=make_generator(CFG["SEED"] + 1),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG["BATCH_SIZE"],
    shuffle=False,
    num_workers=CFG["NUM_WORKERS"],
    pin_memory=(device.type == "cuda"),
    worker_init_fn=seed_worker,
    generator=make_generator(CFG["SEED"] + 2),
)

print("train rows:", len(train_dataset))
print("val rows:", len(val_dataset))
print("test rows:", len(test_dataset))
            


video aug cache invalid: missing rendered files for 2000 samples. rebuilding cache.


Video aug unstable(0.5~1.0s): 100%|##########| 500/500 [01:10<00:00,  7.05it/s]

train rows: 3000
val rows: 100
test rows: 1000


In [5]:
def mixup_multiview_batch(views, labels, alpha=0.2):
    if alpha <= 0:
        return views, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(labels.size(0), device=labels.device)
    mixed_views = [lam * v + (1.0 - lam) * v[index, :] for v in views]
    return mixed_views, labels, labels[index], lam


def train_one_epoch(model, loader, criterion, optimizer, device, mixup_alpha=0.2, mixup_prob=0.5, ema=None):
    model.train()
    train_loss = 0.0
    for views, labels in tqdm(loader, desc="Training", dynamic_ncols=True, ascii=True):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()
        optimizer.zero_grad()

        if mixup_alpha > 0 and np.random.rand() < mixup_prob:
            mixed_views, labels_a, labels_b, lam = mixup_multiview_batch(views, labels, alpha=mixup_alpha)
            outputs = model(mixed_views).view(-1)
            loss = lam * criterion(outputs, labels_a) + (1.0 - lam) * criterion(outputs, labels_b)
        else:
            outputs = model(views).view(-1)
            loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        if ema is not None:
            ema.update(model)
        train_loss += loss.item()

    return train_loss / max(len(loader), 1)


def validate(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation", dynamic_ncols=True, ascii=True):
            views = [v.to(device) for v in views]
            logits = model(views).view(-1)
            probs = torch.sigmoid(logits)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)
    p = np.clip(all_probs, 1e-15, 1 - 1e-15)
    logloss = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))
    acc = np.mean((all_probs > 0.5) == all_labels)
    return float(logloss), float(acc), all_probs, all_labels


def _center_crop_and_resize(x, crop_ratio=0.95):
    b, c, h, w = x.shape
    ch, cw = int(h * crop_ratio), int(w * crop_ratio)
    y1 = (h - ch) // 2
    x1 = (w - cw) // 2
    cropped = x[:, :, y1:y1 + ch, x1:x1 + cw]
    return F.interpolate(cropped, size=(h, w), mode="bilinear", align_corners=False)


def apply_tta_to_views(views, tta_name):
    if tta_name == "none":
        return views
    if tta_name == "hflip":
        return [torch.flip(v, dims=[3]) for v in views]
    if tta_name == "crop95":
        return [_center_crop_and_resize(v, crop_ratio=0.95) for v in views]
    raise ValueError(f"Unknown TTA: {tta_name}")


def predict_probs_with_tta(model, loader, device, tta_names=None, has_labels=False, desc="Inference TTA"):
    tta_names = tta_names or ["none"]
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc, dynamic_ncols=True, ascii=True):
            if has_labels:
                views, labels = batch
                all_labels.extend(labels.numpy())
            else:
                views = batch
            views = [v.to(device) for v in views]
            probs_sum = None
            for tta_name in tta_names:
                tta_views = apply_tta_to_views(views, tta_name)
                logits = model(tta_views).view(-1)
                probs = torch.sigmoid(logits)
                probs_sum = probs if probs_sum is None else (probs_sum + probs)
            all_probs.extend((probs_sum / len(tta_names)).cpu().numpy())
    probs = np.array(all_probs, dtype=np.float64)
    if has_labels:
        return probs, np.array(all_labels, dtype=np.float64)
    return probs


def evaluate_tta_on_dev(model, loader, device, tta_candidates):
    rows = []
    for tta_names in tta_candidates:
        probs, labels = predict_probs_with_tta(
            model, loader, device, tta_names=tta_names, has_labels=True, desc=f"Dev TTA {tta_names}"
        )
        p = np.clip(probs, 1e-15, 1 - 1e-15)
        rows.append({
            "tta_names": tta_names,
            "val_logloss": float(-np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p))),
            "val_acc": float(np.mean((probs > 0.5) == labels)),
        })
    return pd.DataFrame(rows).sort_values("val_logloss").reset_index(drop=True)
            


In [6]:
def train_single_backbone(backbone_name: str):
    config = FlexibleCAFConfig(backbone_name=backbone_name)
    model = MultiViewFlexibleCAF(config).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=CFG["LEARNING_RATE"], weight_decay=CFG["WEIGHT_DECAY"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["EPOCHS"], eta_min=CFG["MIN_LR"])
    ema = ModelEMA(model, EMAConfig(decay=CFG["EMA_DECAY"]))

    safe_name = backbone_name.replace(".", "_").replace("/", "_")
    checkpoint_path = WEIGHT_DIR / f"mv_caf_backbone_test_{safe_name}_v1.2.pt"

    best_logloss = float("inf")
    best_acc = 0.0
    best_epoch = -1
    patience_left = CFG["EARLY_STOPPING_PATIENCE"]
    history = []

    for epoch in range(1, CFG["EPOCHS"] + 1):
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device,
            mixup_alpha=CFG["MIXUP_ALPHA"], mixup_prob=CFG["MIXUP_PROB"], ema=ema
        )
        eval_model = ema.ema_model if CFG["EMA_USE_FOR_EVAL"] else model
        val_logloss, val_acc, _, _ = validate(eval_model, val_loader, device)

        improved = val_logloss < best_logloss
        if improved:
            best_logloss = val_logloss
            best_acc = val_acc
            best_epoch = epoch
            patience_left = CFG["EARLY_STOPPING_PATIENCE"]
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "ema_state_dict": ema.ema_model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "cfg": CFG,
                    "backbone_name": backbone_name,
                    "val_logloss": val_logloss,
                    "val_acc": val_acc,
                },
                checkpoint_path,
            )
        else:
            patience_left -= 1

        scheduler.step()
        history.append({
            "backbone": backbone_name,
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_logloss": float(val_logloss),
            "val_acc": float(val_acc),
            "improved": improved,
            "patience_left": patience_left,
        })
        print(history[-1])

        if patience_left < 0:
            print("early stop:", backbone_name, "epoch=", epoch)
            break

    history_df = pd.DataFrame(history)
    history_df.to_csv(EDA_DIR / f"mv_caf_backbone_test_{safe_name}_history_v1.2.csv", index=False)

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    best_state = checkpoint["ema_state_dict"] if CFG["EMA_USE_FOR_EVAL"] else checkpoint["model_state_dict"]
    model.load_state_dict(best_state)

    tta_df = evaluate_tta_on_dev(model, val_loader, device, CFG["TTA_CANDIDATES"])
    best_tta = tta_df.iloc[0]["tta_names"]
    test_probs = predict_probs_with_tta(model, test_loader, device, tta_names=best_tta, has_labels=False, desc=f"Test {backbone_name}")
    dev_probs, dev_labels = predict_probs_with_tta(model, val_loader, device, tta_names=best_tta, has_labels=True, desc=f"Dev best TTA {backbone_name}")
    p = np.clip(dev_probs, 1e-15, 1 - 1e-15)
    final_logloss = float(-np.mean(dev_labels * np.log(p) + (1 - dev_labels) * np.log(1 - p)))
    final_acc = float(np.mean((dev_probs > 0.5) == dev_labels))

    sub = test_df.copy()
    sub["unstable_prob"] = test_probs
    sub["stable_prob"] = 1.0 - test_probs
    submission_path = SUBMISSION_DIR / f"mv_caf_backbone_test_{safe_name}_v1.2.csv"
    sub.to_csv(submission_path, index=False, encoding="utf-8-sig")

    return {
        "backbone": backbone_name,
        "best_epoch": best_epoch,
        "best_val_logloss": best_logloss,
        "best_val_acc": best_acc,
        "tta_logloss": final_logloss,
        "tta_acc": final_acc,
        "best_tta": best_tta,
        "checkpoint_path": str(checkpoint_path),
        "submission_path": str(submission_path),
        "test_probs": test_probs,
    }
            


In [7]:
results = []
for backbone_name in selected_backbones:
    try:
        result = train_single_backbone(backbone_name)
        results.append(result)
    except Exception as exc:
        print("FAILED:", backbone_name, exc)

result_df = pd.DataFrame([{k: v for k, v in row.items() if k != "test_probs"} for row in results])
result_df = result_df.sort_values(["tta_logloss", "tta_acc"], ascending=[True, False]).reset_index(drop=True)
display(result_df)
result_df.to_csv(EDA_DIR / "mv_caf_backbone_test_v1.2_eval.csv", index=False)
            


Validation: 100%|##########| 7/7 [00:00<00:00,  9.41it/s]


{'backbone': 'efficientnetv2_rw_s', 'epoch': 1, 'train_loss': 0.29367763737335484, 'val_logloss': 0.6905449755259627, 'val_acc': 0.61, 'improved': True, 'patience_left': 5}


Validation: 100%|##########| 7/7 [00:00<00:00, 10.09it/s]


{'backbone': 'efficientnetv2_rw_s', 'epoch': 2, 'train_loss': 0.14520885438678113, 'val_logloss': 0.6562796024046498, 'val_acc': 0.81, 'improved': True, 'patience_left': 5}


Validation: 100%|##########| 7/7 [00:00<00:00, 10.04it/s]


...

Validation: 100%|##########| 7/7 [00:00<00:00, 12.76it/s]


{'backbone': 'deit3_small_patch16_224.fb_in22k_ft_in1k', 'epoch': 17, 'train_loss': 0.14339566978308907, 'val_logloss': 0.21866905683846397, 'val_acc': 0.9, 'improved': False, 'patience_left': -1}
early stop: deit3_small_patch16_224.fb_in22k_ft_in1k epoch= 17


Test deit3_small_patch16_224.fb_in22k_ft_in1k: 100%|##########| 63/63 [00:04<00:00, 15.67it/s]
Dev best TTA deit3_small_patch16_224.fb_in22k_ft_in1k: 100%|##########| 7/7 [00:00<00:00,  9.70it/s]
Training:   0%|          | 0/188 [00:00<?, ?it/s]

FAILED: vit_base_patch14_dinov2.lvd142m Input height (224) doesn't match model (518).


,backbone,best_epoch,best_val_logloss,best_val_acc,tta_logloss,tta_acc,best_tta,checkpoint_path,submission_path
0,convnext_small.fb_in22k_ft_in1k,16,0.097624,0.98,0.097624,0.98,[none],/media/hdd0/whyz/structure-stability/outputs/w...,/media/hdd0/whyz/structure-stability/outputs/s...
1,swin_tiny_patch4_window7_224.ms_in22k_ft_in1k,15,0.143009,0.94,0.111031,0.96,"[none, hflip]",/media/hdd0/whyz/structure-stability/outputs/w...,/media/hdd0/whyz/structure-stability/outputs/s...
2,convnext_tiny.fb_in22k_ft_in1k,13,0.116654,0.94,0.113805,0.95,"[none, hflip, crop95]",/media/hdd0/whyz/structure-stability/outputs/w...,/media/hdd0/whyz/structure-stability/outputs/s...
3,deit3_small_patch16_224.fb_in22k_ft_in1k,11,0.139015,0.94,0.121513,0.96,"[none, hflip]",/media/hdd0/whyz/structure-stability/outputs/w...,/media/hdd0/whyz/structure-stability/outputs/s...
4,efficientnetv2_rw_s,13,0.250585,0.88,0.250585,0.88,[none],/media/hdd0/whyz/structure-stability/outputs/w...,/media/hdd0/whyz/structure-stability/outputs/s...
5,vit_small_patch16_224.augreg_in21k_ft_in1k,13,0.318323,0.89,0.318323,0.89,[none],/media/hdd0/whyz/structure-stability/outputs/w...,/media/hdd0/whyz/structure-stability/outputs/s...
6,vit_base_patch16_224.augreg_in21k_ft_in1k,5,0.549145,0.80,0.549145,0.80,[none],/media/hdd0/whyz/structure-stability/outputs/w...,/media/hdd0/whyz/structure-stability/outputs/s...


In [8]:
assert results, "No successful backbone runs."

top_k = min(3, len(results))
top_df = result_df.head(top_k).copy()
top_names = top_df["backbone"].tolist()
weights = 1.0 / top_df["tta_logloss"].to_numpy()
weights = weights / weights.sum()
print("ensemble backbones:", top_names)
print("weights:", dict(zip(top_names, weights.round(4))))

top_prob_arrays = [next(row["test_probs"] for row in results if row["backbone"] == name) for name in top_names]
ensemble_probs = np.average(np.column_stack(top_prob_arrays), axis=1, weights=weights)

ensemble_sub = test_df.copy()
ensemble_sub["unstable_prob"] = ensemble_probs
ensemble_sub["stable_prob"] = 1.0 - ensemble_probs
ensemble_path = SUBMISSION_DIR / "mv_caf_backbone_test_ensemble_v1.2.csv"
ensemble_sub.to_csv(ensemble_path, index=False, encoding="utf-8-sig")
print("saved:", ensemble_path)
            


ensemble backbones: ['convnext_small.fb_in22k_ft_in1k', 'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k', 'convnext_tiny.fb_in22k_ft_in1k']
weights: {'convnext_small.fb_in22k_ft_in1k': np.float64(0.3654), 'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k': np.float64(0.3212), 'convnext_tiny.fb_in22k_ft_in1k': np.float64(0.3134)}
saved: /media/hdd0/whyz/structure-stability/outputs/submissions/mv_caf_backbone_test/v1.2/mv_caf_backbone_test_ensemble_v1.2.csv
